# 24｜附錄A_自動微分與計算圖

以中央差分檢查解析梯度，並觀察步長過大或過小的誤差。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])


In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.gcf().savefig(folder / f"{name}.svg", bbox_inches="tight")
    plt.show()

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")


## 課堂小實驗

In [ ]:
def f(x, y): return x*x*y + y + 2
x, y, check_eps = 3.0, 4.0, 1e-5
analytic = np.array([2*x*y, x*x+1])
numeric_f = np.array([(f(x+check_eps,y)-f(x-check_eps,y))/(2*check_eps),
                      (f(x,y+check_eps)-f(x,y-check_eps))/(2*check_eps)])

# f 對 x 是二次式，中央差分會特別精準；另用非多項式 g 觀察步長誤差。
def g(v): return np.sin(v) * np.exp(v/10)
g_prime = np.exp(x/10) * (np.cos(x) + 0.1*np.sin(x))
eps_values = np.logspace(-1, -14, 14)
numeric_g = np.array([(g(x+e)-g(x-e))/(2*e) for e in eps_values])
errors = np.abs(numeric_g-g_prime)
best = int(np.argmin(errors))
print('f analytic gradient:', analytic, ' finite difference:', numeric_f)
print('g best epsilon:', eps_values[best], 'error:', errors[best])

## 執行結果圖

In [ ]:
plt.loglog(eps_values, errors, 'o-')
plt.axvline(eps_values[best], color='tab:red', ls='--', label=f'best eps={eps_values[best]:.0e}')
plt.gca().invert_xaxis(); plt.xlabel('epsilon'); plt.ylabel('gradient error')
plt.title('Finite differences fail when epsilon is too large or too small'); plt.legend()
savefig('24_autodiff_demo')
report('autodiff', {'f_analytic': analytic.tolist(), 'f_numeric': numeric_f.tolist(),
                    'g_best_epsilon': eps_values[best], 'g_best_error': errors[best]})

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。